[← GstreamerExp hub](../../index.html) · [README](../../README.md) · [Hypothesis catalog](../../docs/HYPOTHESES.md)

# H10 — Where packet loss breaks SCReAM's quality

**Status:** `supported` · **Source:** Goal 2 boundary behaviour (§7.8)


## Claim

With loss recovery off, a lost packet corrupts VP8 until the next keyframe (60 frames here), so packet loss should break delivery at a low rate. On a fixed 5 Mbps / 20 ms link, as uniform iid loss rises from 0 to 2 percent, frame delivery (frames the viewer reconstructs over frames the camera sent) should stay high at low loss and collapse past some rate, marking a degeneration boundary. PSNR of the frames that do arrive is reported alongside but is not the boundary signal, since the survivors stay clean. If delivery holds across the range, the system tolerates up to 2 percent loss here.

## Predictions

- `usable_delivery_at_zero_loss`
- `delivery_collapses_across_the_loss_range`
- `usable_threshold_found_below_max_loss`

## Verdict

| Outcome | Predicate |
|---|---|
| **Supported when all** | <code>usable_delivery_at_zero_loss</code><br><code>delivery_collapses_across_the_loss_range</code> |
| **Refuted when any** | <code>delivery_robust_across_whole_range</code> |
| **Untested when any** | <code>any_cell_failed</code><br><code>required_metric_missing</code> |


## Findings and Limitations

**Findings**

- Without loss recovery, the system cannot tolerate even 0.1% packet loss. Frame delivery falls from 96% at 0% loss to 36% at 0.1%, 28% at 0.25%, 23% at 0.5%, 18% at 1%, and 17% at 2%. The usable line (80% delivery) is already crossed below 0.1%, so the recovery-off loss boundary is effectively zero.
- {'Both signals degrade, delivery first. PSNR of the frames that do arrive also falls (39.4 dB at 0% loss, then 30.0, 26.4, 23.8, 18.9, 18.3 dB) because corruption propagates into some delivered frames, but the dominant effect is missing frames, not blurry ones': 'a lost packet corrupts VP8 until the next keyframe (6 s here), so each loss costs many frames.'}
- The collapse is real, but its cause is not what it first looked like. This was read as a direct argument for loss recovery; the follow-up H12 re-ran the identical sweep with NACK/RTX/PLI on and found delivery essentially unchanged (largest gain 1.6 points). So the delivery loss is not simple recoverable packet loss -- recovery does not lift the cliff -- and the mechanism (jitter-buffer drop policy under gaps, a pessimistic complete-frame metric, or a SCReAM-under-loss interaction) needs isolation. Next to H9's delay boundary (~200 ms, graceful), the loss boundary is a cliff at zero that recovery, surprisingly, does not move.

**Limitations**

- Recovery is off (nack/pli/fec false), matching H6-H9. This measures the raw loss sensitivity of SCReAM + VP8 with no retransmission; with NACK/RTX on the boundary would move substantially, and that contrast is itself worth a follow-up.
- Single arm (only scream|gcc exist, no bare sender), so the boundary is read off SCReAM's own quality-vs-loss curve.
- Uniform iid loss is swept {0, 0.1, 0.25, 0.5, 1, 2} %; a bursty (Gilbert-Elliott) loss model would degrade differently at the same mean rate.
- {'Workload-specific (realmotion-avi, 1280x1024 MJPEG, 10 fps, 60 s), 4000 kbps ceiling, keyframe interval 60 frames. 3 reps per cell. Boundary signal is frame delivery': 'usable line at 80% delivery, collapse below 50%.'}


## Figures

![Frame delivery vs uniform packet loss (recovery off). The boundary is where delivery falls through the usable line — most frames stop arriving.](results/h10_delivery_by_loss.svg)

*Frame delivery vs uniform packet loss (recovery off). The boundary is where delivery falls through the usable line — most frames stop arriving.*

![PSNR of the frames that do arrive. It stays high because the survivors are clean; the loss damage shows up as missing frames (delivery), not blur.](results/h10_psnr_by_loss.svg)

*PSNR of the frames that do arrive. It stays high because the survivors are clean; the loss damage shows up as missing frames (delivery), not blur.*


## Tables

### `Delivery and PSNR by packet loss`

| metric | 0.0% | 0.1% | 0.25% | 0.5% | 1.0% | 2.0% |
| --- | --- | --- | --- | --- | --- | --- |
| frame delivery | 0.957 | 0.362 | 0.278 | 0.229 | 0.181 | 0.171 |
| PSNR of delivered (dB) | 39.42 | 29.96 | 26.41 | 23.81 | 18.88 | 18.25 |


## Experimental setup

### `h10-loss-boundary`

SCReAM loss-boundary sweep on a fixed 5 Mbps / 20 ms link, recovery off.

**Configurations:** `307` (scream loss=0.0%), `308` (scream loss=0.1%), `309` (scream loss=0.25%), `310` (scream loss=0.5%), `311` (scream loss=1.0%), `312` (scream loss=2.0%) · **Reps:** 3

Spec: `specs/experiments/h10-loss-boundary.yaml` · Record: `runs/experiments/h10-loss-boundary.json` · Run: `python3 tools/run_qdt_sweeps.py  # or experiment.py h10-loss-boundary --resume`

**Status:** 18 of 18 runs completed.


## Required metrics

- `frame_count`
- `encoder_target_kbps`
- `wire_bytes`
- `encoded_bitrate`
- `frame_latency`
- `decoder_errors`
- `decoded_psnr`


## Reproducibility

This notebook is generated from `specs/hypotheses/h10.yaml` and `analysis/hypotheses/results/h10_report.json`. To regenerate:

```sh
python3 analysis/hypotheses/build_reports.py
python3 analysis/hypotheses/build_pages.py
python3 analysis/hypotheses/h10_loss_boundary.py
```

Source: Goal 2 boundary behaviour (§7.8)
